#### Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import torch
np.set_printoptions(precision=2)

from dlgn import DLGN_SP
from utils import set_torchseed
from odt import data_gen_decision_tree
from train import train_dlgn

#### Hyper parameters

In [2]:

config = {
    "input_dim": 100,
    "output_dim": 1,
    "num_layers": 4,
    "num_neurons": 1,
    "odt_depth": 4,

    "beta": 100,
    "learning_rate": 0.2,
    "weight_decay": 0.0,

    "num_data": 2_000_000,
    "num_batches": 1,

    "data_seed": 75218,
    "model_seed": 71000
}

config["hidden_dims"] = [config["num_neurons"]] * config["num_layers"] 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cuda


#### Generate ODT dataset

In [3]:
set_torchseed(config['data_seed'])

(X, y), (w_list, b_list, vals), stats = data_gen_decision_tree(
                                            dim=config["input_dim"], num_levels=config["odt_depth"],
                                            num_data=config["num_data"])

num_data = len(X)
num_train= num_data//2
num_val = num_data//4
num_test = num_data//4
train_data = X[:num_train,:]
train_data_labels = y[:num_train]

val_data = X[num_train:num_train+num_val,:]
val_data_labels = y[num_train:num_train+num_val]

test_data = X[num_train+num_val :,:]
test_data_labels = y[num_train+num_val :]

In [4]:

print(train_data.shape)
print(train_data_labels.shape)

print(test_data.shape)
print(test_data_labels.shape)

print(np.sum(train_data_labels==1))
print(np.sum(train_data_labels==0))

(1000000, 100)
(1000000,)
(500000, 100)
(500000,)
500188
499812


#### DLGN Class

In [5]:
set_torchseed(config["model_seed"])

DLGN_init= DLGN_SP(input_dim=config["input_dim"], 
                   output_dim=config["output_dim"], 
                   hidden_dims=config["hidden_dims"], 
                   beta=config["beta"], 
                   device=device).to(device)
DLGN_init.value_layers.data=torch.ones_like(DLGN_init.value_layers.data)*0.1*(10)

bias = -3
mult = 1e-2

G0 = DLGN_init.gating_layers[0].weight.data
G1 = DLGN_init.gating_layers[1].weight.data
G2 = DLGN_init.gating_layers[2].weight.data
G3 = DLGN_init.gating_layers[3].weight.data

G0.copy_((torch.randint(2,(1,config["input_dim"]))*2 -1.)*0.0)
G1.copy_((torch.randint(2,(1,config["input_dim"]))*2 -1.)*0.0)
G2.copy_((torch.randint(2,(1,config["input_dim"]))*2 -1.)*0.0)
G3.copy_((torch.randint(2,(1,config["input_dim"]))*2 -1.)*0.0)


G0[0,7:15] += bias
G1[0,7:15] += bias
G2[0,7:15] += bias
G3[0,7:15] += bias

G1[0,7:11] *= 1.02
G2[0,7:11] *= 1.02
G3[0,7:11] *= 1.02

G2[0,7] *= 1.02
G3[0,8] *= 1.02

G0[0,7:15] -= bias
G1[0,7:15] -= bias
G2[0,7:15] -= bias
G3[0,7:15] -= bias

G0 *= mult
G1 *= mult
G2 *= mult
G3 *= mult

In [6]:
train_parameter_masks=dict() 
for name,parameter in DLGN_init.named_parameters():
    if "val" in name:
        train_parameter_masks[name]=torch.ones_like(parameter)*0 # Freeze the value layer
    elif "gat" in name and "0" in name:
        train_parameter_masks[name]=torch.ones_like(parameter)*1.
    else:
        train_parameter_masks[name]=torch.ones_like(parameter)*1.          

#### Train

In [ ]:
epochs=1000
DLGN_obj_store, train_losses = train_dlgn(train_data_curr=train_data,
                            val_data_curr=val_data,
                            test_data_curr=test_data,
                            train_labels_curr=train_data_labels,
                            val_labels_curr=val_data_labels,
                            test_labels_curr=test_data_labels,
                            DLGN_obj=deepcopy(DLGN_init),
                            config=config,
                            epochs=epochs,
                            plot=True,
                            val_freeze=False, 
                            parameter_mask=train_parameter_masks
                            )


 83%|████████▎ | 829/1000 [10:34<02:58,  1.04s/it]

#### Visualize

In [ ]:
ew_store_straight_list = []
val_store_straight_list = []

ew_store = np.zeros((len(DLGN_obj_store), config["num_layers"], config["num_neurons"], config["input_dim"]))
val_store = np.zeros((len(DLGN_obj_store), config["num_neurons"]))

for i,model in enumerate(DLGN_obj_store):
    ew = DLGN_obj_store[i].return_gating_functions()
    ew_store[i,0] = ew[0].cpu().numpy()
    ew_store[i,1] = ew[1].cpu().numpy()
    ew_store[i,2] = ew[2].cpu().numpy()
    ew_store[i,3] = ew[3].cpu().numpy()

    val_store[i] = model.value_layers.detach().clone().cpu().numpy()
ew_store_straight_list.append(ew_store)
val_store_straight_list.append(val_store)

In [ ]:
ew_store_straight = np.array(ew_store_straight_list).reshape(-1, config["num_layers"], 1, config["input_dim"])
val_store_straight = np.array(val_store_straight_list).reshape(-1,1)

In [ ]:
print(np.abs(ew_store_straight[-1,:,0,:]).argmax(axis=1))
np.set_printoptions(precision=4)
print(ew_store_straight[0,:,0,7:16])
print(val_store_straight[0])

In [ ]:

k=0
start = 0
end= 400
lines_plot = list(range(0,15)) 
fig, axs = plt.subplots(1,4, sharey="row", figsize=(20,5),  layout="constrained")
for l in range(config["num_layers"]):
    axs[l].set_title(fr"$\ell$={l+1}")
    for i in lines_plot:
        label = fr"$w_\ell.u_{{{i}}}$" if l == 0 else None

        if i>=15:
            axs[l].plot(range(start,end-1),ew_store_straight[start:end-1,l,k,i], "k--", linewidth =1.)
        elif i>=10:
            axs[l].plot(range(start,end-1),ew_store_straight[start:end-1,l,k,i], label=label, ls="dashed")
        else:
            axs[l].plot(range(start,end-1),ew_store_straight[start:end-1,l,k,i], label=label, ls="solid")

    axs[l].plot([start,end-2],[0,0],'k--')

# Collect handles and labels from all axes
handles_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
handles, labels = [sum(lol, []) for lol in zip(*handles_labels)]

# Add a single legend for the figure
fig.legend(handles, labels, loc='outside right center', ncol=1)
fig.suptitle("Parameter components along ODT node vectors vs training epoch")

fig, axs = plt.subplots(1,4,sharey="row",figsize=(20,5),  layout="constrained")
for l in range(config["num_layers"]):
    # plt.ylim([-0.001,0.001])
    axs[l].set_title(fr"$\ell$={l+1}")
    # plt.yscale("symlog", linthresh=0.01)
    # plt.xscale("log")
    # axs[l].set_xscale("log")
    for i in lines_plot:
        # plt.plot([start,end],[0.02,0.02],ls="dotted", lw=1)
        label = fr"$\Delta w_\ell.u_{{{i}}}$" if l == 0 else None
        if i>=15:
            axs[l].plot(range(start,end-1),ew_store_straight[start+1:end,l,k,i]-ew_store_straight[start:end-1,l,k,i], "k--", linewidth =1. )
        elif i>=10:
            axs[l].plot(range(start,end-1),ew_store_straight[start+1:end,l,k,i]-ew_store_straight[start:end-1,l,k,i], label=label, ls="dashed")
        else:
            axs[l].plot(range(start,end-1),ew_store_straight[start+1:end,l,k,i]-ew_store_straight[start:end-1,l,k,i], label=label, ls="solid")

    axs[l].plot([start,end-2],[0,0],'k--')
# Collect handles and labels from all axes
handles_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
handles, labels = [sum(lol, []) for lol in zip(*handles_labels)]
fig.suptitle("Change in parameter components along ODT node vectors vs training epoch")

fig.legend(handles, labels, loc='outside right center', ncol=1)

plt.show()